In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

# Cantat-Gaudin+2020 catalog

In [ ]:
import sys
sys.path.insert(0, '..')
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="CantatGaudin2020", catalog_type="cluster")
tabs = cd.get_tables_from_vizier(row_limit=-1, save=False)

In [ ]:
df = tabs[0].to_pandas()
df_mem = tabs[1].to_pandas()
df_mem.head()

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({"Cluster": "Name",
                        "Source": "GaiaDR3",
                        "Proba": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem.head()

In [ ]:
from wakai.catalog import get_tois

tois = get_tois(clobber=False)
# tois.columns.tolist()
tois.shape

In [ ]:
tois[tois.Comments.apply(lambda x: 'young' in str(x))]

# cross-match TOI with Gaia DR3 using `astroquery.xmatch`

https://astroquery.readthedocs.io/en/latest/xmatch/xmatch.html

In [ ]:
import pandas as pd

# output of cross_match_TOI_with_GaiaDR3.ipynb
matches = pd.read_csv('../data/TOIs_GaiaDR3_xmatch_position.csv')
matches.head()

In [ ]:
matches.shape

In [ ]:
matches.loc[:, 'GaiaDR3'] = matches['DR3Name'].apply(lambda x: int(x.split()[-1]))
final = pd.merge(df_mem, matches, on='GaiaDR3')
final.head()

In [ ]:
final.shape

In [ ]:
new_cols = "TOI	TIC ID	Name	GaiaDR3	Prob	TESS Disposition	TFOPWG Disposition	Comments	TESS Mag	TESS Mag err	Source_1	Detection	Epoch (BJD)	Epoch (BJD) err	Period (days)	Period (days) err	Duration (hours)	Duration (hours) err	Depth (mmag)	Depth (mmag) err	Depth (ppm)	Depth (ppm) err	Planet Radius (R_Earth)	Planet Radius (R_Earth) err	Planet Insolation (Earth Flux)	Planet Equil Temp (K)	Planet SNR	Stellar Distance (pc)	Stellar Distance (pc) err	Stellar Eff Temp (K)	Stellar Eff Temp (K) err	Stellar log(g) (cm/s^2)	Stellar log(g) (cm/s^2) err	Stellar Radius (R_Sun)	Stellar Radius (R_Sun) err	Stellar Metallicity	Stellar Metallicity err	Stellar Mass (M_Sun)	Stellar Mass (M_Sun) err	Sectors	Date TOI Alerted (UTC)	Date TOI Updated (UTC)	Date Modified	ra_deg	dec_deg	Pipeline Signal ID	RA	Dec	PM RA (mas/yr)	PM RA err (mas/yr)	PM Dec (mas/yr)	PM Dec err (mas/yr)	DR3Name	RAdeg	DEdeg	errHalfMaj	errHalfMin	errPosAng	SolID	Source_2	RandomI	e_RAdeg	e_DEdeg	e_Plx	RPlx	PM	e_pmRA	e_pmDE	RADEcor	RAPlxcor	RApmRAcor	RApmDEcor	DEPlxcor	DEpmRAcor	DEpmDEcor	PlxpmRAcor	PlxpmDEcor	pmRApmDEcor	NAL	NAC	NgAL	NbAL	gofAL	chi2AL	epsi	sepsi	Solved	APF	nueff	MatchObsA	Nper	amax	MatchObs	IPDgofha	IPDgofhp	IPDfmp	IPDfow	RUWE	Dup	o_Gmag	FG	e_FG	RFG	Gmag_y	e_Gmag	o_BPmag	FBP	e_FBP	RFBP	e_BPmag	o_RPmag	FRP	e_FRP	RFRP	e_RPmag	E(BP/RP)	NBPcont	NBPblend	NRPcont	NRPblend	Mode	BP-G	G-RP	e_RV	n_RV	o_RV	o_RVd	RVNper	RVS/N	RVgof	RVchi2	RVTdur	RVamp	RVtempTeff	RVtemplogg	RVtemp[Fe/H]	Vatmparam	vbroad	e_Vbroad	o_Vbroad	GRVSmag	e_GRVSmag	o_GRVSmag	RVSS/N	VarFlag	PQSO	PGal	PSS	Teff	b_Teff	B_Teff	logg	b_logg	B_logg	[Fe/H]	b_[Fe/H]	B_[Fe/H]	Dist	b_Dist	B_Dist	A0	b_A0	B_A0	AG	b_AG	B_AG	E(BP-RP)	b_E(BP-RP)	B_E(BP-RP)	Lib	RAJ2000	DEJ2000	e_RAJ2000	e_DEJ2000	RADEcorJ2000	Previous CTOI	Master	SG1A	SG1B	SG2	SG3	SG4	SG5	ESM	TSM	Predicted Mass (M_Earth)	Time Series Observations	Spectroscopy Observations	Imaging Observations".split('\t')

In [ ]:
final[new_cols].to_csv(f'../data/{cd.catalog_name}/TOIs_in_{cd.catalog_name}.csv', index=False)

In [ ]:
final[new_cols].sort_values(by='TOI', ascending=False)